# Nutrition5k — All Experiments (Unattended Run)

Runs Exp 1 → 2 → 3 → 4 sequentially with automatic VRAM/RAM management.

**Start → Run All → Walk Away**

| Exp | Description | Data | Est. time |
|-----|-------------|------|-----------|
| 1 | Per-gram, side-angle | rgb_train + rgb_test | ~8–12 h |
| 2 | Direct, side-angle | rgb_train + rgb_test | ~8–12 h |
| 3 | Depth as 4th channel | depth_train + depth_test | ~2–4 h |
| 4 | Mass regressor + cal pipeline | depth + rgb_test | ~2–4 h |


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, sys

if os.path.isdir('/content/Nutrition5k'):
    subprocess.run(['git', '-C', '/content/Nutrition5k', 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/T0MYYY/Nutrition5k.git',
                    '/content/Nutrition5k', '-b', 'master', '-q'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/Nutrition5k', '-q'], check=True)

if '/content/Nutrition5k' not in sys.path:
    sys.path.insert(0, '/content/Nutrition5k')
import importlib; importlib.invalidate_caches()

CONFIG_DIR = '/content/Nutrition5k/configs/colab'
DRIVE      = '/content/drive/MyDrive/nutrition5k'
print(f'Ready. CONFIG_DIR={CONFIG_DIR}')

In [ ]:
import torch

subprocess.run(['apt-get', 'install', '-qq', '-y', 'zstd'],
               check=True, capture_output=True)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print(f'GPU: {_p.name}   VRAM: {_p.total_memory // 1024**3} GB')
    VRAM_GB = _p.total_memory // 1024**3
else:
    VRAM_GB = 0
print(f'VRAM_GB = {VRAM_GB}')

In [ ]:
import gc, csv as _csv, threading, shutil, yaml
import numpy as np
from PIL import Image

from nutrition5k_pkg.train import run, load_config, _build_model
from nutrition5k_pkg.evaluate import evaluate
from nutrition5k_pkg.data.metadata import load_dish_metadata, load_official_split
from nutrition5k_pkg.data.transforms import get_val_transform
from nutrition5k_pkg.models.multitask_head import MultitaskNutritionNet
from nutrition5k_pkg.metrics import compute_mae_report, print_results_table

_vol_csv = f'{DRIVE}/data/metadata/volume_estimates.csv'
assert os.path.isfile(_vol_csv), (
    f'volume_estimates.csv not found: {_vol_csv}\nRun 00_prepare_data.ipynb first.')
print('Prerequisites OK')

In [ ]:
def _configure(cfg_path, base_bs, base_lr, base_head_lr):
    with open(cfg_path) as _f:
        _cfg = yaml.safe_load(_f)
    if VRAM_GB >= 80:   _bs = base_bs
    elif VRAM_GB >= 40: _bs = base_bs // 2
    elif VRAM_GB >= 24: _bs = base_bs // 4
    else:               _bs = base_bs // 8
    _scale = _bs / base_bs
    _cfg['training']['batch_size'] = _bs
    _cfg['training']['lr']         = round(base_lr * _scale, 7)
    _cfg['training']['head_lr']    = round(base_head_lr * _scale, 7)
    with open(cfg_path, 'w') as _f:
        yaml.dump(_cfg, _f)
    _name = os.path.basename(cfg_path).replace('.yaml', '')
    print(f'{_name:35s}  batch={_bs}  '
          f'lr={_cfg["training"]["lr"]:.2e}  head_lr={_cfg["training"]["head_lr"]:.2e}')

_configure(f'{CONFIG_DIR}/exp1_portion_independent.yaml', 256, 3e-4, 1e-3)
_configure(f'{CONFIG_DIR}/exp2_direct_prediction.yaml',   256, 3e-4, 1e-3)
_configure(f'{CONFIG_DIR}/exp3_depth_channel.yaml',       256, 3e-4, 1e-3)
_configure(f'{CONFIG_DIR}/exp4_volume_scalar.yaml',       256, 3e-4, 1e-3)

---
## Phase A — Side-Angle RGB

Experiments 1 and 2. Loads `rgb_train.tar.zst` + `rgb_test.tar.zst` in parallel (~6 min).

In [ ]:
_DST    = '/dev/shm'
_marker = '/dev/shm/side_angles'
os.makedirs(_marker, exist_ok=True)

def _extract(archive):
    _src = f'{DRIVE}/data/{archive}'
    print(f'  Extracting {archive} ...')
    subprocess.run(f'zstd -d --stdout "{_src}" | tar -x -C {_DST}',
                   shell=True, check=True)
    print(f'  Done: {archive}')

_n_existing = sum(1 for d in os.scandir(_marker) if d.is_dir())
_test_id    = open(f'{DRIVE}/data/splits/rgb_test_ids.txt').readline().strip()
_todo = []
if _n_existing <= 100:                             _todo.append('rgb_train.tar.zst')
if not os.path.isdir(f'{_marker}/{_test_id}'):    _todo.append('rgb_test.tar.zst')

if _todo:
    _threads = [threading.Thread(target=_extract, args=(a,), daemon=True) for a in _todo]
    for _t in _threads: _t.start()
    for _t in _threads: _t.join()
else:
    print('Side-angle data already in /dev/shm.')

print(f'Ready: {sum(1 for d in os.scandir(_marker) if d.is_dir())} dishes in {_marker}')

### Exp 1 — Per-Gram Prediction (side-angle)

In [ ]:
print('=' * 60)
print('EXP 1: Per-Gram Prediction')
print('=' * 60)
_ckpt = f'{DRIVE}/checkpoints/exp1/best.pt'
if os.path.isfile(_ckpt):
    best_ckpt_exp1 = _ckpt
    print(f'Checkpoint found, skipping training: {best_ckpt_exp1}')
else:
    best_ckpt_exp1 = run(f'{CONFIG_DIR}/exp1_portion_independent.yaml')
    print(f'Exp1 checkpoint: {best_ckpt_exp1}')


In [ ]:
if 'best_ckpt_exp1' not in dir(): best_ckpt_exp1 = f'{DRIVE}/checkpoints/exp1/best.pt'
_cfg = load_config(f'{CONFIG_DIR}/exp1_portion_independent.yaml')
_m   = _build_model(_cfg)
_m.load_state_dict(torch.load(best_ckpt_exp1, map_location='cpu'))
evaluate(_m, f'{CONFIG_DIR}/exp1_portion_independent.yaml', f'{DRIVE}/results/exp1')
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Exp1 eval complete.')

### Exp 2 — Direct Prediction (side-angle)

In [ ]:
print('=' * 60)
print('EXP 2: Direct Prediction')
print('=' * 60)
_ckpt = f'{DRIVE}/checkpoints/exp2/best.pt'
if os.path.isfile(_ckpt):
    best_ckpt_exp2 = _ckpt
    print(f'Checkpoint found, skipping training: {best_ckpt_exp2}')
else:
    best_ckpt_exp2 = run(f'{CONFIG_DIR}/exp2_direct_prediction.yaml')
    print(f'Exp2 checkpoint: {best_ckpt_exp2}')


In [ ]:
if 'best_ckpt_exp2' not in dir(): best_ckpt_exp2 = f'{DRIVE}/checkpoints/exp2/best.pt'
_cfg = load_config(f'{CONFIG_DIR}/exp2_direct_prediction.yaml')
_m   = _build_model(_cfg)
_m.load_state_dict(torch.load(best_ckpt_exp2, map_location='cpu'))
evaluate(_m, f'{CONFIG_DIR}/exp2_direct_prediction.yaml', f'{DRIVE}/results/exp2')
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Exp2 eval complete.')

In [ ]:
print('Releasing Phase A data from /dev/shm ...')
shutil.rmtree('/dev/shm/side_angles', ignore_errors=True)
gc.collect()
_df = subprocess.run(['df', '-h', '/dev/shm'], capture_output=True, text=True).stdout
print(_df.strip())
print('Phase A complete.')

---
## Phase B — Overhead RGB-D

Experiments 3 and 4. Loads `depth_train/test` + `rgb_test` in parallel (~3–4 min).

> `rgb_test` is re-extracted here for Exp 4 Stage 2 combined evaluation.

In [ ]:
os.makedirs('/dev/shm/realsense_overhead', exist_ok=True)
os.makedirs('/dev/shm/side_angles', exist_ok=True)

def _extract_if_needed(archive, marker_dir, min_dishes):
    _n = (sum(1 for d in os.scandir(marker_dir) if d.is_dir())
          if os.path.isdir(marker_dir) else 0)
    if _n > min_dishes:
        print(f'  {archive}: already extracted ({_n} dishes)'); return
    _src = f'{DRIVE}/data/{archive}'
    print(f'  Extracting {archive} ...')
    subprocess.run(f'zstd -d --stdout "{_src}" | tar -x -C /dev/shm',
                   shell=True, check=True)
    print(f'  Done: {archive}')

_todos = [
    ('depth_train.tar.zst', '/dev/shm/realsense_overhead', 50),
    ('depth_test.tar.zst',  '/dev/shm/realsense_overhead', 50),
    ('rgb_test.tar.zst',    '/dev/shm/side_angles',        0),
]
_threads = [threading.Thread(target=_extract_if_needed, args=_t, daemon=True) for _t in _todos]
for _t in _threads: _t.start()
for _t in _threads: _t.join()

print(f'Overhead dishes:        {sum(1 for d in os.scandir("/dev/shm/realsense_overhead") if d.is_dir())}')
print(f'Side-angle (test only): {sum(1 for d in os.scandir("/dev/shm/side_angles") if d.is_dir())}')

### Exp 3 — Depth as 4th Channel (overhead RGB-D)

In [ ]:
print('=' * 60)
print('EXP 3: Depth Channel')
print('=' * 60)
_ckpt = f'{DRIVE}/checkpoints/exp3/best.pt'
if os.path.isfile(_ckpt):
    best_ckpt_exp3 = _ckpt
    print(f'Checkpoint found, skipping training: {best_ckpt_exp3}')
else:
    best_ckpt_exp3 = run(f'{CONFIG_DIR}/exp3_depth_channel.yaml')
    print(f'Exp3 checkpoint: {best_ckpt_exp3}')


In [ ]:
if 'best_ckpt_exp3' not in dir(): best_ckpt_exp3 = f'{DRIVE}/checkpoints/exp3/best.pt'
_cfg = load_config(f'{CONFIG_DIR}/exp3_depth_channel.yaml')
_m   = _build_model(_cfg)
_m.load_state_dict(torch.load(best_ckpt_exp3, map_location='cpu'))
evaluate(_m, f'{CONFIG_DIR}/exp3_depth_channel.yaml', f'{DRIVE}/results/exp3')
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Exp3 eval complete.')

### Exp 4 — Volume Scalar Pipeline (overhead RGB-D)

Stage 1: train mass regressor. Stage 2: combine with Exp 1 per-gram model for calorie prediction.

In [ ]:
# Re-extract overhead data if /dev/shm was cleared (e.g. after runtime restart)
def _re_extract(archive, dest, min_dishes=0):
    import subprocess as _sp
    _n = sum(1 for _d in os.scandir(dest) if _d.is_dir()) if os.path.isdir(dest) else 0
    if _n > min_dishes:
        print(f'  {archive}: already extracted ({_n} dishes)'); return
    os.makedirs(dest, exist_ok=True)
    _src = f'{DRIVE}/data/{archive}'
    print(f'  Extracting {archive} ...')
    _sp.run(f'zstd -d --stdout "{_src}" | tar -x -C /dev/shm', shell=True, check=True)
    print(f'  Done.')
_re_extract('depth_train.tar.zst', '/dev/shm/realsense_overhead', 50)
_re_extract('depth_test.tar.zst',  '/dev/shm/realsense_overhead', 0)
_re_extract('rgb_test.tar.zst',    '/dev/shm/side_angles', 0)

print('=' * 60)
print('EXP 4: Volume Scalar Pipeline')
print('=' * 60)
_ckpt = f'{DRIVE}/checkpoints/exp4/best.pt'
if os.path.isfile(_ckpt):
    best_ckpt_exp4 = _ckpt
    print(f'Checkpoint found, skipping training: {best_ckpt_exp4}')
else:
    best_ckpt_exp4 = run(f'{CONFIG_DIR}/exp4_volume_scalar.yaml')
    print('Exp4 mass regressor checkpoint:', best_ckpt_exp4)


In [ ]:
if 'best_ckpt_exp1' not in dir(): best_ckpt_exp1 = f'{DRIVE}/checkpoints/exp1/best.pt'
if 'best_ckpt_exp4' not in dir(): best_ckpt_exp4 = f'{DRIVE}/checkpoints/exp4/best.pt'

_device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_overhead_tfm = get_val_transform(pre_resized=False)  # overhead images are not pre-resized
_side_tfm     = get_val_transform(pre_resized=True)   # side-angle frames are pre-resized

# Per-gram model from Exp 1
_pg_tasks = ['cal_per_g', 'fat_per_g', 'carb_per_g', 'protein_per_g']
_pg_model = MultitaskNutritionNet(tasks=_pg_tasks).to(_device)
_pg_model.load_state_dict(torch.load(best_ckpt_exp1, map_location=_device))
_pg_model.eval()

# Mass regressor from Exp 4
_exp4_cfg   = load_config(f'{CONFIG_DIR}/exp4_volume_scalar.yaml')
_mass_model = _build_model(_exp4_cfg).to(_device)
_mass_model.load_state_dict(torch.load(best_ckpt_exp4, map_location=_device))
_mass_model.eval()

# Metadata + test split + volume map
_meta = load_dish_metadata(_exp4_cfg['data']['cafe1_csv'], _exp4_cfg['data'].get('cafe2_csv'))
_, _, _test_ids = load_official_split(
    _exp4_cfg['data']['train_ids_txt'], _exp4_cfg['data']['test_ids_txt'])
with open(f'{DRIVE}/data/metadata/volume_estimates.csv') as _f:
    _vol_map = {_r['dish_id']: float(_r['volume_cm3']) for _r in _csv.DictReader(_f)}

_OVERHEAD = '/dev/shm/realsense_overhead'
_SIDE     = '/dev/shm/side_angles'
_preds_cal, _gt_cal = [], []

with torch.no_grad():
    for _did in _test_ids:
        _rgb_path = os.path.join(_OVERHEAD, _did, 'rgb.png')
        if not os.path.isfile(_rgb_path) or _did not in _vol_map:
            continue
        _oh_img = _overhead_tfm(Image.open(_rgb_path).convert('RGB')).unsqueeze(0).to(_device)
        _vol_t  = torch.tensor([[_vol_map[_did]]], dtype=torch.float32).to(_device)
        _mass   = _mass_model(_oh_img, _vol_t).item()

        _frames_dir = os.path.join(_SIDE, _did, 'frames')
        if not os.path.isdir(_frames_dir):
            continue
        _frames = sorted(x for x in os.listdir(_frames_dir)
                         if x.lower().endswith(('.jpg', '.jpeg', '.png')))
        if not _frames:
            continue
        _si_img = _side_tfm(Image.open(os.path.join(_frames_dir, _frames[0])).convert('RGB')).unsqueeze(0).to(_device)
        _cal_pg = _pg_model(_si_img)['cal_per_g'].item()
        _preds_cal.append(_mass * _cal_pg)
        _gt_cal.append(_meta[_did]['calories'])

print(f'Evaluated {len(_preds_cal)} dishes')
_res = {'calories': compute_mae_report(np.array(_preds_cal), np.array(_gt_cal))}
print_results_table(_res)

_out = f'{DRIVE}/results/exp4'
os.makedirs(_out, exist_ok=True)
with open(os.path.join(_out, 'predictions.csv'), 'w', newline='') as _f:
    _w = _csv.writer(_f)
    _w.writerow(['pred_calories', 'gt_calories'])
    for _p, _g in zip(_preds_cal, _gt_cal):
        _w.writerow([_p, _g])
print('Exp4 results saved to:', _out)

del _pg_model, _mass_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('All experiments complete.')

In [ ]:
from google.colab import runtime
runtime.unassign()